# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_ccd.csv |
| Cluster curve rows | 316 |
| Items | 29 |
| Train items | 20 |
| Test items | 9 |
| Global Beta alpha | 0.868080 |
| Global Beta beta | 1.211064 |
| Global train MSE | 0.02756449 |
| Proxy label CSV | clustered_curve_proxy_labels_ccd.csv |
| Proxy-labeled held-out rows | 111 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1089 | 0.1481 | 0.0768 | 0.2374 | 0.0009 |
| Pooled empirical median curve | 0.1166 | 0.1653 | 0.0859 | 0.2669 | 0.0098 |
| Global Beta CDF | 0.1195 | 0.1596 | 0.0887 | 0.2714 | 0.0007 |
| Cluster-count-bucket Beta CDF | 0.1335 | 0.1801 | 0.1024 | 0.3042 | 0.0217 |
| Linear cumulative spend | 0.1417 | 0.1817 | 0.1292 | 0.3125 | -0.0821 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | 23.14% |
| RMSE improvement vs linear | 18.50% |
| P90 absolute-error improvement vs linear | 24.05% |
| Linear bias | -0.0821 |
| Duration-bucket Beta bias | 0.0009 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.765177 | 1.182536 |
| 366-730d | 0.931801 | 1.237124 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.822843 | 1.381631 |
| 7-12 clusters | 0.899527 | 1.187444 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 24 | 52 | 24 | 28 | 0 |
| 0.10 | spending too slowly / time-overrun risk | 21 | 11 | 11 | 0 | 10 |
| 0.15 | spending too quickly / budget-overrun risk | 15 | 40 | 15 | 25 | 0 |
| 0.15 | spending too slowly / time-overrun risk | 13 | 9 | 9 | 0 | 4 |
| 0.25 | spending too quickly / budget-overrun risk | 3 | 17 | 3 | 14 | 0 |
| 0.25 | spending too slowly / time-overrun risk | 7 | 3 | 3 | 0 | 4 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.9994 | 17 | 94 |
| fast_spend_proxy | Linear cumulative spend | 1.0000 | 17 | 94 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9987 | 17 | 94 |
| slow_spend_proxy | Linear cumulative spend | 0.9875 | 17 | 94 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 39 | 17 | 22 | 0.436 | 1.000 | 0.234 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 24 | 17 | 7 | 0.708 | 1.000 | 0.074 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 15 | 15 | 0 | 1.000 | 0.882 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 10 | 10 | 0 | 1.000 | 0.588 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 3 | 3 | 0 | 1.000 | 0.176 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 1 | 1 | 0 | 1.000 | 0.059 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 64 | 17 | 47 | 0.266 | 1.000 | 0.500 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 52 | 17 | 35 | 0.327 | 1.000 | 0.372 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 40 | 17 | 23 | 0.425 | 1.000 | 0.245 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 22 | 17 | 5 | 0.773 | 1.000 | 0.053 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 17 | 17 | 0 | 1.000 | 1.000 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 12 | 12 | 0 | 1.000 | 0.706 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 2 | 2 | 0 | 1.000 | 0.118 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 34 | 17 | 17 | 0.500 | 1.000 | 0.181 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 21 | 17 | 4 | 0.810 | 1.000 | 0.043 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 13 | 13 | 0 | 1.000 | 0.765 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 11 | 11 | 0 | 1.000 | 0.647 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 7 | 7 | 0 | 1.000 | 0.412 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 6 | 6 | 0 | 1.000 | 0.353 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 3 | 3 | 0 | 1.000 | 0.176 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 18 | 13 | 5 | 0.722 | 0.765 | 0.053 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 11 | 11 | 0 | 1.000 | 0.647 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 9 | 9 | 0 | 1.000 | 0.529 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 7 | 7 | 0 | 1.000 | 0.412 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 3 | 3 | 0 | 1.000 | 0.176 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 3 | 3 | 0 | 1.000 | 0.176 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 0 | 0 | 0 | 0.000 | 0.000 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.